# NumCompute — Quick Start

This notebook shows how to load CSV data, preprocess numerical and categorical columns, combine the steps into a Pipeline, and use a simple estimator. The example uses `assets/students.csv` from the project repository.

## 1. Import Packages

Import the NumCompute functions used in the demo. 

In [1]:
import os
os.chdir("..")
import numpy as np
from numcompute.io import load_csv, load_csv_chunks
from numcompute.pipeline import Pipeline, FeatureUnion
from numcompute.preprocessing import ColumnSelector, SimpleImputer, StandardScaler, OneHotEncoder, MinMaxScaler

## 2. Load CSV Data

`load_csv()` returns a 2D NumPy array and the column names. Numerical values are initially stored as strings, so they can be handled together with categorical values.

In [2]:
data, columns = load_csv("assets/students.csv")
print("Columns:", columns)
print("Shape:", data.shape)
print("Data type:", data.dtype)
print("First three rows:")
print(data[:3])

Columns: ['StudentID', 'Age', 'GPA', 'City', 'Course']
Shape: (10, 5)
Data type: <U11
First three rows:
[['1001' '20' '5.5' 'Adelaide' 'Computing']
 ['1002' '22' '6.0' 'Sydney' 'DataScience']
 ['1003' '' '4.5' 'Melbourne' 'Computing']]


### Read the CSV in Chunks

`load_csv_chunks()` returns the same as load_csv, but reads the data in smaller chunks.

In [3]:

print("Read CSV in Chunks:")

for i, (chunk, columns) in enumerate(
    load_csv_chunks("assets/students.csv", chunk_size=4), start=1
):
    print(f"\nChunk {i}:")
    print("Shape:", chunk.shape)
    print(chunk)


Read CSV in Chunks:

Chunk 1:
Shape: (4, 5)
[['1001' '20' '5.5' 'Adelaide' 'Computing']
 ['1002' '22' '6.0' 'Sydney' 'DataScience']
 ['1003' '' '4.5' 'Melbourne' 'Computing']
 ['1004' '19' '' 'Adelaide' 'Math']]

Chunk 2:
Shape: (4, 5)
[['1005' '25' '6.5' '' 'Computing']
 ['1006' '21' '5.0' 'Canberra' '']
 ['1007' '' '5.8' 'Sydney' 'Computing']
 ['1008' '23' '' 'Melbourne' 'DataScience']]

Chunk 3:
Shape: (2, 5)
[['1009' '24' '6.2' 'Adelaide' 'Math']
 ['1010' '20' '5.7' 'Brisbane' 'Computing']]


## 3. Select Numerical and Categorical Columns

`ColumnSelector` selects columns by their names. We use `Age` and `GPA` as numerical features and `City` and `Course` as categorical features. 

In [4]:
numeric_selector = ColumnSelector(columns, ["Age", "GPA"])
numeric = numeric_selector.fit_transform(data)
categorical_selector = ColumnSelector(columns, ["City", "Course"])
categorical = categorical_selector.fit_transform(data)

## 4. Fill Missing Values

`SimpleImputer` replaces missing values. We fill numerical values with `0` and convert them to `float`. For categorical values, we use `"Unknown"`.

In [5]:
numeric_imputer = SimpleImputer(fill_value=0, dtype=float)
numeric_filled = numeric_imputer.fit_transform(numeric)
categorical_imputer = SimpleImputer(fill_value="Unknown", dtype=str)
categorical_filled = categorical_imputer.fit_transform(categorical)

## 5. Scale Numerical Data And Encode Categorical Data

`StandardScaler` uses the mean and standard deviation of each column. `MinMaxScaler` scales each column to a range (default `[0, 1]`). Both are applied to the same filled data for comparison. `OneHotEncoder` converts each category to binary (0/1) features. 

In [6]:
standard_scaler = StandardScaler()
standard_scaled = standard_scaler.fit_transform(numeric_filled)
minmax_scaler = MinMaxScaler()
minmax_scaled = minmax_scaler.fit_transform(numeric_filled)
encoder = OneHotEncoder()
categorical_encoded = encoder.fit_transform(categorical_filled)

## 6. Create a Complete Preprocessing Pipeline

`Pipeline` runs transformations in order. `FeatureUnion` runs the numerical and categorical branches on the same input and combines their outputs.


In [7]:
pipe = Pipeline([
    ("preprocess", FeatureUnion([
        ("numeric", Pipeline([
            ("select", ColumnSelector(columns, ["Age", "GPA"])),
            ("impute", SimpleImputer(0, float)),
            ("scale", StandardScaler())
        ])),
        ("categorical", Pipeline([
            ("select", ColumnSelector(columns, ["City", "Course"])),
            ("impute", SimpleImputer("Unknown", str)),
            ("encode", OneHotEncoder())
        ]))
    ]))
])
result = pipe.fit_transform(data)

## 7. Pipeline on New Data

After fitting a Pipeline, use `transform()` for new data. This uses the means, standard deviations, and categories learned during fitting. New data must have the same column order as the training data.

In [8]:
new_data = np.array([
    ["1011", "21", "5.8", "Adelaide", "Computing"],
    ["1012", "", "6.1", "Sydney", "Math"]
])

new_result = pipe.transform(new_data)

print("New data shape:", new_result.shape)
print("Transformed new data:")
print(np.round(new_result, 3))
assert new_result.shape[1] == result.shape[1]

New data shape: (2, 12)
Transformed new data:
[[ 0.405  0.551  1.     0.     0.     0.     0.     0.     1.     0.
   0.     0.   ]
 [-1.96   0.68   0.     0.     0.     0.     1.     0.     0.     0.
   1.     0.   ]]


## 8. Optional: Add an Estimator

NumCompute does not include a machine-learning model. The `Pipeline` can still use a model supplied by the user, as long as it implements `fit(data, y)` and `predict(data)` and is the last step.

## 9. Statistics and Metrics

Here are examples of other functions in NumCompute. `stats.py` works with numerical arrays. `metrics.py` compares true and predicted values.

In [9]:
from numcompute.stats import mean, std, quantile, histogram
from numcompute.metrics import accuracy, mse
print("Feature means:")
print(np.round(mean(result), 3))
print("\nFeature standard deviations:")
print(np.round(std(result), 3))

gpa = numeric_filled[:, 1]

print("\nGPA Quantiles (25%, 50%, 75%):")
print(quantile(gpa, [0.25, 0.5, 0.75]))

counts, edges = histogram(gpa, bins=5)
print("\nGPA Histogram:")
print("Counts:", counts)
print("Bin edges:", np.round(edges, 2))

y_true = np.array([1, 0, 1, 1])
y_pred = np.array([1, 1, 1, 0])
print("\nAccuracy:", accuracy(y_true, y_pred))

actual = np.array([2.0, 4.0, 6.0])
predicted = np.array([2.5, 3.5, 6.0])
print("MSE:", mse(actual, predicted))

Feature means:
[ 0.  -0.   0.3  0.1  0.1  0.2  0.2  0.1  0.5  0.2  0.2  0.1]

Feature standard deviations:
[1.    1.    0.458 0.3   0.3   0.4   0.4   0.3   0.5   0.4   0.4   0.3  ]

GPA Quantiles (25%, 50%, 75%):
[4.625 5.6   5.95 ]

GPA Histogram:
Counts: [2 0 0 2 6]
Bin edges: [0.  1.3 2.6 3.9 5.2 6.5]

Accuracy: 0.5
MSE: 0.16666666666666666



## 10. Sorting and Searching

This module provides different functions for sorting and searching numerical data. It includes three sorting methods: `stable_sort()`, `quick_sort()`, and `merge_sort()`.

`topk()` finds the k largest or smallest values, while `top_p()` selects a given proportion of values. 

`quickselect()` finds the kth smallest value using a zero-based index.

`binary_search()` searches for a target value in a sorted array and returns its index and whether the value was found.


In [10]:
from numcompute.sort_search import stable_sort, quick_sort, merge_sort, topk, quickselect, binary_search

sortdata = np.array([8, 3, 5, 3, 9, 1, 7])

print("Original data:", sortdata)
print("\nStable Sort:", stable_sort(sortdata))
print("Quick Sort:", quick_sort(sortdata))
print("Merge Sort:", merge_sort(sortdata))
print("\nTop 3 largest:", topk(sortdata, 3))
print("\nThird smallest value:", quickselect(sortdata, 2))
sorted_data = stable_sort(sortdata)
index, found = binary_search(sorted_data, 5)
print("\nBinary Search for 5:")
print("Index:", index)
print("Found:", found)

Original data: [8 3 5 3 9 1 7]

Stable Sort: [1 3 3 5 7 8 9]
Quick Sort: [1 3 3 5 7 8 9]
Merge Sort: [1 3 3 5 7 8 9]

Top 3 largest: [9 8 7]

Third smallest value: 3

Binary Search for 5:
Index: 3
Found: True



## 11. Optimisation

This module provides functions for calculating numerical derivatives using finite differences.

`gradient()` calculates the gradient of a function with a single output. It shows how the function changes with respect to each input variable.

`jacobian()` calculates the Jacobian matrix of a function with multiple outputs. Each row represents an output, and each column represents an input variable.

Both functions use central finite differences to estimate derivatives, with a default step size of `1e-5`.


In [11]:
from numcompute.optim import gradient, jacobian

def f(x): 
    return x[0]**2 + 3*x[1]**2

funcdata = np.array([2.0, 3.0])

print("Input:", funcdata)
print("\nGradient:", np.round(gradient(f, funcdata), 4))

def F(x):
    return np.array([x[0] + x[1], x[0] * x[1]])

print("\nJacobian:")
print(np.round(jacobian(F, funcdata), 4))


Input: [2. 3.]

Gradient: [ 4. 18.]

Jacobian:
[[1. 1.]
 [3. 2.]]
